In [1]:
# ---- Install FLOPs tool (Colab) ----
!pip install thop

# ---- Imports ----
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torch.amp import autocast, GradScaler
from thop import profile
import numpy as np
import random

In [2]:

# ---- Reproducibility ----
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

# ---- Constants ----
BATCH_SIZE = 16
LR = 0.001
EPOCHS = 1
USE_AMP = True
OPTIMIZERS = ["SGD", "Adam"]
MODELS = ["resnet18", "resnet50"]
DEVICES = ["cpu", "cuda"] if torch.cuda.is_available() else ["cpu"]

# ---- Dataset ----
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

total_len = len(dataset)
train_len = int(0.7 * total_len)
val_len = int(0.1 * total_len)
rem_len = total_len - train_len - val_len

train_set, val_set, _ = random_split(dataset, [train_len, val_len, rem_len])


100%|██████████| 26.4M/26.4M [00:01<00:00, 13.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 273kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.50MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.0MB/s]


In [3]:
# ---- Dataloaders ----
def get_loaders(device):
    pin = True if device == "cuda" else False
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)
    return train_loader, val_loader, test_loader

# ---- Model Builder ----
def get_model(name, device):
    if name == "resnet18":
        model = models.resnet18(weights=None)
    else:
        model = models.resnet50(weights=None)

    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(device)

In [4]:
# ---- Optimizer ----
def get_optimizer(name, params):
    if name == "SGD":
        return optim.SGD(params, lr=LR, momentum=0.9)
    return optim.Adam(params, lr=LR)

# ---- Train One Epoch ----
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    correct, total = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        with autocast(device, enabled=(device == "cuda" and USE_AMP)):
            out = model(x)
            loss = criterion(out, y)

        if device == "cuda":
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        _, preds = out.max(1)
        correct += preds.eq(y).sum().item()
        total += y.size(0)

    return 100.0 * correct / total

In [5]:
# ---- Evaluation ----
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            _, preds = out.max(1)
            correct += preds.eq(y).sum().item()
            total += y.size(0)

    return 100.0 * correct / total

In [6]:
# ---- FLOPs Calculation ----
def compute_flops(model, device):
    dummy = torch.randn(1, 3, 224, 224).to(device)
    flops, params = profile(model, inputs=(dummy,), verbose=False)
    return flops

# ---- Experiments ----
for device in DEVICES:
    print("\n" + "=" * 110)
    print(f"DEVICE: {device.upper()}")
    print("=" * 110)

    train_loader, val_loader, test_loader = get_loaders(device)

    for model_name in MODELS:
        for opt_name in OPTIMIZERS:

            print("\n" + "-" * 100)
            print(f"Model: {model_name} | Optimizer: {opt_name}")
            print("-" * 100)

            model = get_model(model_name, device)
            optimizer = get_optimizer(opt_name, model.parameters())
            criterion = nn.CrossEntropyLoss()
            scaler = GradScaler(device, enabled=(device == "cuda" and USE_AMP))

            # FLOPs
            flops = compute_flops(model, device)
            print(f"FLOPs: {flops / 1e9:.3f} GFLOPs")

            # Training Time
            start_time = time.time()

            for epoch in range(EPOCHS):
                train_acc = train_one_epoch(
                    model, train_loader, optimizer, criterion, scaler, device
                )
                val_acc = evaluate(model, val_loader, device)

                print(f"Epoch [{epoch+1}/{EPOCHS}] | "
                      f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

            end_time = time.time()
            train_time_ms = (end_time - start_time) * 1000

            # Test Accuracy
            test_acc = evaluate(model, test_loader, device)

            print(f"Final Test Accuracy: {test_acc:.2f}%")
            print(f"Total Training Time: {train_time_ms:.2f} ms")


DEVICE: CPU

----------------------------------------------------------------------------------------------------
Model: resnet18 | Optimizer: SGD
----------------------------------------------------------------------------------------------------
FLOPs: 1.824 GFLOPs
Epoch [1/1] | Train Acc: 77.68% | Val Acc: 86.95%
Final Test Accuracy: 86.56%
Total Training Time: 3269880.52 ms

----------------------------------------------------------------------------------------------------
Model: resnet18 | Optimizer: Adam
----------------------------------------------------------------------------------------------------
FLOPs: 1.824 GFLOPs
Epoch [1/1] | Train Acc: 82.68% | Val Acc: 85.53%
Final Test Accuracy: 85.46%
Total Training Time: 3090956.71 ms

----------------------------------------------------------------------------------------------------
Model: resnet50 | Optimizer: SGD
----------------------------------------------------------------------------------------------------
FLOPs: 4.132